# Step 1 — Speech-to-text transcripts (Whisper large-v3)

Transcribes every train (769) and test (216) clip with `openai/whisper-large-v3` on GPU and saves
`train_transcripts.csv` / `test_transcripts.csv` to `/kaggle/working` for the grammar-scoring model.

Clips are 45–60 s, longer than Whisper's 30 s window, so the HF pipeline's chunked long-form mode is used.

In [ ]:
import glob, os
import pandas as pd, torch
from transformers import pipeline

# Locate the dataset root regardless of the exact mount path
ROOT = os.path.dirname(glob.glob('/kaggle/input/**/train.csv', recursive=True)[0])
OUT = '/kaggle/working'
print(ROOT, os.listdir(ROOT))
train = pd.read_csv(f'{ROOT}/train.csv')
test = pd.read_csv(f'{ROOT}/test.csv')
print(train.shape, test.shape)
train.head()

In [ ]:
assert torch.cuda.is_available(), 'Enable a GPU accelerator'
asr = pipeline(
    'automatic-speech-recognition',
    model='openai/whisper-large-v3',
    torch_dtype=torch.float16,
    device='cuda:0',
    chunk_length_s=30,  # clips are 45-60 s -> chunked long-form decoding
)
GEN = {'language': 'en', 'task': 'transcribe'}

In [ ]:
def audio_path(split, name):
    """CSV filenames may or may not carry the .wav extension."""
    name = str(name)
    return f'{ROOT}/{split}/{name if name.endswith(".wav") else name + ".wav"}'

def transcribe(df, split):
    col = df.columns[0]  # first column holds the file name
    paths = [audio_path(split, n) for n in df[col]]
    missing = [p for p in paths if not os.path.exists(p)]
    assert not missing, f'{len(missing)} missing files, e.g. {missing[:3]}'
    outs = asr(paths, batch_size=4, generate_kwargs=GEN)
    out = df.copy()
    out['transcript'] = [o['text'].strip() for o in outs]
    out.to_csv(f'{OUT}/{split}_transcripts.csv', index=False)
    return out

train_t = transcribe(train, 'train')
test_t = transcribe(test, 'test')

In [ ]:
# Sanity check: empty transcripts and word-count distribution
for name, d in [('train', train_t), ('test', test_t)]:
    wc = d['transcript'].str.split().str.len()
    print(name, 'empty:', (wc == 0).sum(), '| words mean/min/max:', wc.mean().round(1), wc.min(), wc.max())
train_t.sample(5, random_state=0)